# 00 - Setup and data load

**Purpose**: verify the environment is healthy, load every dataframe the
other notebooks need, and write them to a parquet cache so the
downstream notebooks open fast.

Run this notebook **first**. After the final cell finishes, open any of
the 01-06 notebooks in order -- they all read from the cache produced
here and never touch the SQLite database directly.


## 1. Environment health check

Runs ``endpoint_ck_analysis.doctor.doctor()`` which prints a table of required
pieces (Python version, packages, database file, writable cache,
region-prior source). Everything should be ``[OK]``.

If anything is ``[FAIL]``:
  - Missing package? The "detail" column gives the ``pip install`` command.
  - Database file not found? Copy ``connectome.db`` into
    ``endpoint_ck_analysis/_bundled_data/`` (the folder you see next to this
    notebook's parent) or set ``ENDPOINT_CK_ANALYSIS_DB`` to its location.
  - Still stuck? Open ``TROUBLESHOOTING.md`` in the project root.


In [ ]:
from endpoint_ck_analysis.doctor import doctor # entry point that runs every diagnostic check and prints a status table
doctor() # prints the [OK]/[FAIL]/[INFO] table; returns True when there are no blockers


**What you just saw.** Every required check should show `[OK]`. Optional
items (like `jupyterlab` if you're using VS Code) may show `[INFO]` --
that's fine and expected. Any `[FAIL]` line blocks the analysis; the
detail column on that line tells you what to do (usually a single
`pip install ...` command). If everything is green, you can move on.


## 2. Load all dataframes

``load_all`` reads ``connectome.db`` (location resolved by
``config.get_db_path()``), runs every filter/aggregation/pivot from
Section 6 of the original notebook, and returns a :class:`LoadedData`
container with everything downstream code needs.

On first run this takes ~30 seconds (SQL + aggregation). Subsequent
runs read the cache in under a second unless you delete it or pass
``use_cache=False`` to force a rebuild.


In [ ]:
from endpoint_ck_analysis.data_loader import load_all # one-shot loader that returns a LoadedData with every dataframe downstream notebooks consume

data = load_all(use_cache=False, write_cache=True) # use_cache=False forces a fresh DB rebuild; write_cache=True saves parquet so notebooks 01-08 don't re-touch the DB. Set use_cache=True after the first run for fast reloads.

print()
print('Base dataframes:') # AKDdf and friends are the per-reach / per-region tables; everything else is derived from these
for name in ['AKDdf', 'FKDdf', 'ACDUdf', 'ACDGdf', 'FCDUdf', 'FCDGdf']: # iterate by name string so the loop also prints the variable name alongside the shape
    print(f'  {name}: {getattr(data, name).shape}') # getattr(data, name) pulls the dataframe attribute matching the loop variable; .shape gives (rows, columns)

print()
print('Connectomics wide pivots:') # one row per subject, one column per region_hemi; produced by helpers.connectivity.pivot_connectivity
for name in ['ACDUdf_wide', 'ACDGdf_wide', 'FCDUdf_wide', 'FCDGdf_wide']:
    print(f'  {name}: {getattr(data, name).shape}')

print()
print(f'Matched subjects (both kinematics and connectomics): {list(data.matched_subjects)}') # the analyzable cohort: only subjects in this list participate in PLS or any cross-block analysis


**What you just saw.** Every dataframe shape should be non-empty. The
first block is the base data (`AKDdf` is the all-mice kinematic
dataframe; `F*` versions are filtered to the matched cohort, `*Udf` are
ungrouped/atomic-region, `*Gdf` are eLife-grouped). The wide pivots
are subject x region_hemi matrices used by every connectomics analysis
downstream. The `Matched subjects` line is the most consequential:
only mice with both kinematics AND connectomics data appear there, and
every cross-block question in 04 / 07 is answered from that cohort.

If any shape is `(0, 0)` or the matched-subjects list is empty, the
upstream pipeline didn't produce what we expected -- usually means the
wrong DB is configured, recent backfills weren't applied, or the
imaging-parameter filter excluded everything. Check `connectome.db`
before continuing.


## 3. Quick-look previews

Spot-check that the data resembles what you expect before running the
analytical notebooks. Uncomment any lines that would be useful to see.


In [ ]:
from IPython.display import display                                                          # display(): force-renders any object as cell output. Plain `df.head()` only shows when it's the LAST line of the cell, so for multi-preview cells we wrap each in display() instead.
# Uncomment any of the lines below to peek at the loaded data. Each one renders a different dataframe so you can sanity-check shape and contents before continuing.
# display(data.AKDdf.head())                                                                 # first 5 rows of All Kinematic Data: per-reach kinematic features for every analyzable phase
# display(data.FCDGdf_wide)                                                                  # Filtered Connectivity Data Grouped (eLife) -- wide subjects x region_hemi matrix; matched-subject rows only
# display(data.AKDdf_agg_contact.head())                                                     # first 5 rows of the (subject, phase, contact_group) aggregation: mean/std/median/q25/q75 per kinematic feature
pass                                                                                          # 'pass' is a no-op placeholder; required because Python disallows an empty cell body when all real lines are commented out. With display(), the position of pass no longer matters: every uncommented display() call still prints.


## Next

- **01_connectivity_pca.ipynb** -- PCA on the connectivity matrix.
- **02_kinematic_pca.ipynb** -- PCA on kinematics, one per phase, with sign alignment.
- **03_kinematic_clustering.ipynb** -- hierarchical clustering of kinematic features.
- **04_pls_variants.ipynb** -- PLS across connectivity and kinematics (injury / deficit / recovery).
- **05_lmm_phase_effects.ipynb** -- Linear mixed models for per-feature phase effects.
- **06_pellet_validation.ipynb** -- manual vs algorithmic pellet scoring agreement.
- **99_figure_gallery.ipynb** -- all saved figures in one place.
